# Musanga V4 — NB03 | DSM structural predictors and paired RGB/DSM preflight

**Stage:** executable feature engineering and modelling-readiness QA (not a trained model). This notebook prepares a scientifically controlled RGB + SfM-DSM experiment using the **current 78-field-plot / 34-circle V4 reference** and frozen 3-fold window manifest. It does **not** report performance improvements until matched ablations have been trained and validated.

**Run cells in order.** Inputs `Final/Data`, V3_1 and NB00/NB01 contracts are read-only. Outputs are written exclusively to `Final/Outputs/v4/03_dsm_predictor_preflight/`.

Scientific caution: these DSMs are photogrammetric **canopy-surface elevations, not ground-normalized CHMs**. Local prominence, roughness and convexity are relative *surface-shape* descriptors, not tree heights. The DSMs must pass valid-pixel, registration and acquisition matching checks before cross-modal model training.

Planned ablation: paired M0 RGB; M1 RGB + local prominence; M2 RGB + prominence + roughness + convexity. Compare on the **same folds and jointly valid supports**. Multispectral/LAS are deferred until acquisition-specific QA.


In [ ]:
# 00 — Environment, frozen input manifests and experimental settings
%pip -q install rasterio geopandas pyogrio scipy matplotlib pandas numpy torch segmentation-models-pytorch
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from scipy.ndimage import uniform_filter, gaussian_filter
from shapely.geometry import box
import matplotlib.pyplot as plt
from IPython.display import display
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except ImportError:
    pass

ROOT_CANDIDATES=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                 Path('/content/drive/MyDrive/Musanga/Final')]
ROOT=next((x for x in ROOT_CANDIDATES if x.is_dir()),None)
assert ROOT is not None, f'Final folder not found: {ROOT_CANDIDATES}'
DATA=ROOT/'Data'
OUT=ROOT/'Outputs/v4/03_dsm_predictor_preflight'
OUT.mkdir(parents=True,exist_ok=True)
NB00_AUDIT=ROOT/'Outputs/v4/00_input_audit/00_rgb_dsm_pair_audit.csv'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT_FILE=REF_DIR/'01_field78_reference_contract.json'
assert NB00_AUDIT.is_file(), 'Run V4 NB00 DSM audit first'
assert CONTRACT_FILE.is_file(), 'Run V4 NB01 final reference sections first'
CONTRACT=json.loads(CONTRACT_FILE.read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2'
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096 and set(SPLITS.cv_fold)=={0,1,2}
PAIRS=pd.read_csv(NB00_AUDIT)
assert len(PAIRS)==6 and PAIRS.rgb.is_unique
CHIP=256
# Window radii in metres. These must be reviewed against crown scale and
# are FIXED a priori for the preflight, not tuned on held-out references.
PROMINENCE_RADIUS_M=6.0
ROUGHNESS_RADIUS_M=3.0
CONVEXITY_RADIUS_M=2.0
MAX_QA_WINDOWS_PER_MOSAIC=6
MIN_JOINT_VALID_FRACTION=0.95
print('NB03 DSM preflight:',OUT)
print('Manifest split totals:')
display(SPLITS.groupby(['cv_fold','site','split']).size().rename('n_windows').reset_index())


## 01 — Validate the exact six RGB/DSM associations

NB00 chooses **candidates** by ROI coverage. Its candidate ranking is not proof of alignment. This cell fails closed on missing paths, missing CRS, insufficient extent, and ambiguous matches. For known MOS survey groups, the RGB and DSM survey IDs must agree. Manually review `00_rgb_dsm_pair_audit.csv` before approving a pairing. No automatic fallback to another site's DSM.


In [ ]:
# 01 — Conservative pair registry. Approve only after registration QA.
def mosaic_key(name):
    v=str(name).casefold().replace(' ','_')
    if 'baego' in v:return 'Baego'
    if 'mos' in v:
        for k in ('1_2','3_4','5_6'):
            if k in v:return 'MOS_'+k
        return 'MOS_unknown'
    if 'yangambi' in v or 'ybjoseph' in v or 'ybforgeo' in v or 'carlier' in v:return 'Yangambi'
    if 'yoko' in v:return 'Yoko'
    return 'unknown'

REGISTRY={}
for rec in PAIRS.itertuples(index=False):
    rgb=Path(rec.rgb_path);dsm=Path(rec.dsm_path)
    assert rgb.is_file() and dsm.is_file(),(rgb,dsm)
    assert mosaic_key(rgb.name)==mosaic_key(dsm.name),(rgb.name,dsm.name)
    assert float(rec.roi_overlap_fraction)>=.95,(
        'DSM ROI overlap <95%; matched comparison needs separate coverage handling',
        rgb.name,rec.roi_overlap_fraction)
    with rasterio.open(rgb) as a,rasterio.open(dsm) as b:
        assert a.crs is not None and b.crs is not None
        assert a.count>=3 and b.count>=1
        assert a.crs.to_epsg()==32635,(rgb,a.crs)
        px=(abs(a.transform.a),abs(a.transform.e))
        assert 0<min(px)<100,(rgb,px)
        REGISTRY[rgb.name]=dict(rgb=str(rgb),dsm=str(dsm),
            site=mosaic_key(rgb.name),px=px,transform=a.transform,
            shape=(a.height,a.width),dsm_count=b.count,
            rgb_crs=str(a.crs),dsm_crs=str(b.crs))
display(pd.DataFrame([dict(rgb=k,dsm=Path(v['dsm']).name,
      rgb_resolution_m=v['px'][0],dsm_bands=v['dsm_count'])
      for k,v in REGISTRY.items()]))
(OUT/'03_pair_registry_provisional.json').write_text(json.dumps(
    {k:{z:(list(q) if isinstance(q,tuple) else str(q) if z=='transform' else q)
        for z,q in v.items()} for k,v in REGISTRY.items()},indent=2))
print('WARNING: provisional pairs. Bbox overlap alone does NOT certify registration.')


## 02 — Local surface geometry (no false CHM)

On each requested RGB tile plus DSM processing halo, sample the DSM onto the RGB grid using bilinear interpolation, with an explicit nodata mask. Derive local detrended prominence, detrended roughness and curvature-like convexity. Use valid-aware smoothing so DSM nodata/edges do not create artificial peaks. **Only valid interior pixels** can enter paired RGB/DSM evaluation.

Filters are specified in metres; raw DSM elevation is excluded from the first structural ablation because arbitrary elevation offsets can encode sites/surveys.


In [ ]:
# 02 — Windowed, halo-safe structural feature extraction
from contextlib import ExitStack
def _mean_valid(a,valid,size):
    f=valid.astype(np.float32)
    num=uniform_filter(np.where(valid,a,0.).astype(np.float32),size=size,mode='nearest')
    den=uniform_filter(f,size=size,mode='nearest')
    return num/np.maximum(den,1e-6),den

def _odd_pixels(radius_m,pixel_m):
    r=max(1,int(np.ceil(radius_m/pixel_m)))
    return 2*r+1

def feature_tile(rgb_path,dsm_path,win):
    """Return 6xHxW [RGB3, prominence, roughness, convexity], joint valid HxW.
    Requires actual pixel registration to be separately approved before training.
    """
    with ExitStack() as stack:
        rgb=stack.enter_context(rasterio.open(rgb_path))
        dsm=stack.enter_context(rasterio.open(dsm_path))
        assert rgb.crs is not None and dsm.crs is not None
        h,w=int(win.height),int(win.width)
        assert h>0 and w>0 and int(win.row_off)>=0 and int(win.col_off)>=0
        assert win.row_off+h<=rgb.height and win.col_off+w<=rgb.width
        step=min(abs(rgb.transform.a),abs(rgb.transform.e))
        max_radius=max(PROMINENCE_RADIUS_M,ROUGHNESS_RADIUS_M,
                       CONVEXITY_RADIUS_M)
        halo=int(np.ceil(2.0*max_radius/step))+4
        r0=max(0,int(win.row_off)-halo);c0=max(0,int(win.col_off)-halo)
        r1=min(rgb.height,int(win.row_off)+h+halo)
        c1=min(rgb.width,int(win.col_off)+w+halo)
        expanded=Window(c0,r0,c1-c0,r1-r0)
        tr=rgb.window_transform(expanded)
        rgb_raw=rgb.read([1,2,3],window=expanded,masked=True)
        if not all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3]):
            raise ValueError(f'RGB is not uint8: {rgb_path}: {rgb.dtypes}')
        rgb_valid=(rgb.dataset_mask(window=expanded)>0)&(
            ~np.ma.getmaskarray(rgb_raw).any(axis=0))
        rgb_data=np.asarray(rgb_raw.filled(0),dtype=np.float32)/255.
        vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,transform=tr,
            width=int(expanded.width),height=int(expanded.height),
            resampling=Resampling.bilinear,src_nodata=dsm.nodata,
            nodata=np.nan))
        arr=vrt.read(1,masked=True)
        z=np.asarray(arr.filled(np.nan),dtype=np.float32)
        dsm_valid=(~np.ma.getmaskarray(arr))&np.isfinite(z)
        valid=rgb_valid&dsm_valid
        z0=np.where(dsm_valid,z,0.).astype(np.float32)
        prominence_size=_odd_pixels(PROMINENCE_RADIUS_M,step)
        rough_size=_odd_pixels(ROUGHNESS_RADIUS_M,step)
        convex_size=_odd_pixels(CONVEXITY_RADIUS_M,step)
        base,base_support=_mean_valid(z0,dsm_valid,prominence_size)
        local,rough_support=_mean_valid(z0,dsm_valid,rough_size)
        local2,_=_mean_valid(z0*z0,dsm_valid,rough_size)
        smooth,conv_support=_mean_valid(z0,dsm_valid,convex_size)
        prominence=z0-base
        roughness=np.sqrt(np.maximum(0.,local2-local**2))
        convexity=z0-smooth
        # Exclude filters with inadequate valid support near DSM boundaries.
        valid &= (base_support>=.95)&(rough_support>=.95)&(conv_support>=.95)
        y=int(win.row_off)-r0;x=int(win.col_off)-c0
        sl=np.s_[y:y+h,x:x+w]
        x6=np.concatenate([rgb_data[:,sl[0],sl[1]],
            prominence[sl][None],roughness[sl][None],
            convexity[sl][None]],axis=0).astype(np.float32)
        joint=valid[sl].copy()
        x6[:,~joint]=0.
        assert x6.shape==(6,h,w) and joint.shape==(h,w)
        assert np.isfinite(x6).all()
        return x6,joint

FEATURE_NAMES=('R','G','B','local_prominence_m',
               'surface_roughness_m','local_convexity_m')
print('Available features:',FEATURE_NAMES)


## 03 — Frozen manifest sampling and coverage QA

Sample only existing final NB01 training/validation windows; do not regenerate folds. Report per-mosaic DSM valid-pixel fractions on RGB-valid ROI and representative images. The inspection is intentionally bounded to avoid scanning massive UAV orthomosaics on Drive.


In [ ]:
# 03 — Low-I/O matched-support QA, existing fold assignments only
ROI=gpd.read_file(DATA/'Ortho_RGB'/'ROI.shp').to_crs('EPSG:32635')
qa=[]
examples={}
for name,entry in REGISTRY.items():
    rows=SPLITS[(SPLITS.mosaic==name)&SPLITS.split.eq('val')].sort_values(
        ['cv_fold','row','col']).head(MAX_QA_WINDOWS_PER_MOSAIC)
    if len(rows)==0:
        rows=SPLITS[(SPLITS.mosaic==name)&SPLITS.split.eq('train')].head(
            MAX_QA_WINDOWS_PER_MOSAIC)
    for rec in rows.itertuples(index=False):
        win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
        f,valid=feature_tile(entry['rgb'],entry['dsm'],win)
        tr=rasterio.windows.transform(win,entry['transform'])
        footprint=box(*rasterio.windows.bounds(win,entry['transform']))
        nearby=ROI[ROI.intersects(footprint)].geometry
        roi_mask=rasterize(((g,1) for g in nearby),
            out_shape=valid.shape,transform=tr,fill=0,dtype='uint8').astype(bool)
        support=roi_mask
        fraction=float((valid&support).sum()/support.sum()) if support.any() else np.nan
        qa.append(dict(mosaic=name,site=rec.site,fold=int(rec.cv_fold),
            row=int(rec.row),col=int(rec.col),split=rec.split,
            n_roi_pixels=int(support.sum()),n_joint_valid=int((valid&support).sum()),
            joint_valid_fraction=fraction,
            prominence_p95=float(np.percentile(f[3][valid],95)) if valid.any() else np.nan,
            roughness_p95=float(np.percentile(f[4][valid],95)) if valid.any() else np.nan))
        if name not in examples and valid.any():examples[name]=(f,valid)
QA=pd.DataFrame(qa)
assert len(QA)>0
QA.to_csv(OUT/'03_dsm_joint_valid_window_QA.csv',index=False)
display(QA.groupby('mosaic').agg(n_windows=('row','size'),
    min_joint_valid=('joint_valid_fraction','min'),
    median_joint_valid=('joint_valid_fraction','median'),
    n_ge95=('joint_valid_fraction',lambda a:int((a>=.95).sum()))).round(3))
for name,(features,valid) in examples.items():
    fig,ax=plt.subplots(1,4,figsize=(15,3.5),layout='constrained')
    ax[0].imshow(features[:3].transpose(1,2,0));ax[0].set_title('RGB')
    for k,title in [(3,'Prominence (m)'),(4,'Roughness (m)'),(5,'Convexity (m)')]:
        im=ax[k-2].imshow(np.ma.masked_where(~valid,features[k]),cmap='viridis')
        ax[k-2].set_title(title)
        fig.colorbar(im,ax=ax[k-2],shrink=.65)
    for a in ax:a.axis('off')
    fig.suptitle(name)
    fig.savefig(OUT/f'03_dsm_features_{Path(name).stem}.png',dpi=130)
    plt.show();plt.close(fig)
print('These small samples do NOT certify co-registration or whole-ROI completeness.')


## 04 — Model input contract and fold-safe normalization

This is a **modelling interface**, not training. The baseline uses ImageNet-pretrained ResNet18 U-Net and masked BCE. DSM requires fold-specific normalization fitted only on eligible outer-training windows and a six-channel first convolution. Do not copy global DSM statistics from held-out plots.

The next training stage will use M0/M1/M2 on identical eligible windows with the original train/val roles, a blocked inner checkpoint-selection split, and matched held-out evaluation. It must use the same 50%-overlap Hann-blended probabilities as V4.009 before claiming gains.


In [ ]:
# 04 — Persist feature definition and matching requirements (no model fitting yet)
EXPERIMENT={
 'schema':'musanga_v4_nb03_dsm_preflight_1',
 'base_model':'V4.007 RGB ResNet18 U-Net; V4.009 50%-overlap evaluation',
 'reference_contract':str(CONTRACT_FILE),
 'window_manifest':str(CONTRACT['cv_manifest']),
 'folds':[0,1,2],
 'field_plots_total':78,'circles_total':34,
 'rgb_channels':list(FEATURE_NAMES[:3]),
 'dsm_channels':list(FEATURE_NAMES[3:]),
 'feature_windows_m':{'prominence':PROMINENCE_RADIUS_M,
                      'roughness':ROUGHNESS_RADIUS_M,
                      'convexity':CONVEXITY_RADIUS_M},
 'feature_order':list(FEATURE_NAMES),
 'candidate_models':{
   'M0':['R','G','B'],
   'M1':['R','G','B','local_prominence_m'],
   'M2':list(FEATURE_NAMES)},
 'run_status':'PREFLIGHT_ONLY_NOT_TRAINED',
 'required_before_training':[
   'Manually approve per-survey RGB/DSM co-registration and vertical offsets',
   'Confirm valid pixel overlap over all held-out supports, not sampled bounding boxes',
   'Check distribution of DSM feature scales and non-finite/nodata support',
   'Fit DSM normalization using only each outer fold training pixels',
   'Train on identical frozen spatial splits; inner selection must be spatially blocked',
   'Evaluate RGB baseline and DSM models on identical joint-valid held-out supports']}
(OUT/'03_dsm_experiment_contract.json').write_text(json.dumps(EXPERIMENT,indent=2))
print('Saved:',OUT/'03_dsm_experiment_contract.json')
print('Preflight complete. Do NOT interpret these checks as trained DSM-model results.')
